## STEP4 - Join Ners

In [1]:
#!/usr/bin/env python
# coding: utf-8

# In[1]:


#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
NER Output Alignment Script
Created on Tue Mar 16 09:49:30 2021
Updated for clarity, optimization, and robustness.
"""

import os
import pandas as pd
import sys
from tqdm import tqdm


# In[2]:


import os

working_dir = os.getcwd()
print(working_dir)

parts = working_dir.split("/")
if parts[-1].startswith("pipeline"): ### BUG SOLVED MAY 2025
    pipeline = working_dir
else:
    pipeline ="/".join(parts[:-1])
print(pipeline)


# In[3]:


#base = "/home/ematos/phd/pt/INITIAL_NERs/"
sys.path.append(pipeline)

# Configure directories
try:
    from config.configsite import * #filesdir, linguakitdir, inputdir, outputdir, basedir
except ImportError:
    print("Error: 'configsite' module not found. Ensure it is correctly configured.")
    sys.exit(1)

filesdir = f"{pipeline}/{dataset}_PROCESSED/"
lista_path = filesdir + "/files_list.csv"

lista = pd.read_csv(lista_path)
lista.head()

outputdir=pipeline+"INITIAL_NERS/"+resultsdir+"/"
print(f"output dir = {outputdir}") # new AT april 2025


# In[6]:


def log_debug(message):
    """Writes debug messages to the debug file if debugging is enabled."""
    if DEBUG:
        debug_file.write(message + "\n")


# In[7]:


def align_words(data1, data2, bio_col1, bio_col2, file_col, line_col, pos_col, window=6):
        """
        Aligns words between two NER datasets and returns a merged DataFrame.

        Args:
            data1 (pd.DataFrame): First dataset.
            data2 (pd.DataFrame): Second dataset.
            bio_col1 (str): Column name for BIO tags in the first dataset.
            bio_col2 (str): Column name for BIO tags in the second dataset.
            file_col (str): Column name for the file identifier.
            line_col (str): Column name for the line identifier.
            pos_col (str): Column name for the position identifier.
            window (int): Sliding window size for handling mismatches.

        Returns:
            pd.DataFrame: Aligned DataFrame.
        """
        i, j = 0, 0
        len1, len2 = len(data1), len(data2)
        merged_data = []

        while i < len1:
            word1 = data1.iloc[i]["WORD"]
            word2 = data2.iloc[j]["WORD"] if j < len2 else None
            log_debug(f"** {i}: word1={word1}  {j}: word2={word2}")

            if word1 == word2:
                merged_data.append({
                    "WORD": word1,
                    bio_col1: data1.iloc[i][bio_col1],
                    bio_col2: data2.iloc[j][bio_col2] if j < len2 else "O",
                    "FILE": data1.iloc[i][file_col],
                    "LINE": data1.iloc[i][line_col],
                    "POSITION": data1.iloc[i][pos_col]
                })
                i += 1
                j += 1
            else:
                # Handle misalignment with a sliding window
                next_words1 = data1.iloc[i:i + window]["WORD"].tolist()
                next_words2 = data2.iloc[j:j + window]["WORD"].tolist()

                if word1 in next_words2:
                    j += next_words2.index(word1)
                elif word2 in next_words1:
                    i += next_words1.index(word2)
                else:
                    merged_data.append({
                        "WORD": word1,
                        bio_col1: data1.iloc[i][bio_col1],
                        bio_col2: "O",
                        "FILE": data1.iloc[i][file_col],
                        "LINE": data1.iloc[i][line_col],
                        "POSITION": data1.iloc[i][pos_col]
                    })
                    i += 1

        return pd.DataFrame(merged_data)


# In[11]:


# Iterate through each file in the list
debug = True
DEBUG = False

for _, row in tqdm(lista.iterrows(), total=len(lista), desc="Processing files", unit="file"):
    filename = row.get("filename")
    basename = row.get("basename")

    print(f"\nProcessing {filename}")
 

    # Paths and filenames
    #FILENAMES = ["output.allen.csv", "output.linguakit", "output.dbpediaNER"]
    FILENAMES = ["output.allen.csv", "output.linguakit", "output.dbpediaNER"] # novo 20 june

    # Load input files
    try:
        filename_allen = f"{outputdir}{basename}/{FILENAMES[0]}"
        if debug:
            print(f"file Allen = {filename_allen}")
            
        data_allen = pd.read_csv(filename_allen, sep=",")
        data_allen.rename(columns={"BIO_ALLEN": "BIO1"}, inplace=True)
    except Exception as e:
        print(f"\n **** Error = {e}\nCONTINUING")
        continue
    

    try:
        filename_linguakit= f"{outputdir}{basename}/{FILENAMES[1]}"
        if debug:
            print(f"file Linguakit = {filename_linguakit}")
            
        data_linguakit = pd.read_csv(filename_linguakit, sep=",")
        data_linguakit.rename(columns={"BIO-LINGUAKIT": "BIO2"}, inplace=True)
        print(data_linguakit.head(3))
    except Exception as e:
        print(f"\n **** Error = {e}\nCONTINUING")
        continue

    try:
        auxx = f"{outputdir}{basename}/{FILENAMES[2]}"
        if debug:
            print(f"file dbpedia = {auxx}")
        data_dbpedia = pd.read_csv(auxx, sep=",")
        data_dbpedia.rename(columns={"BIO": "BIO3", "Token": "WORD"}, inplace=True)
 
    except Exception as e:
        auxx = f"{outputdir}{basename}/{FILENAMES[2]}"
        #data_dbpedia = pd.read_csv(f"{outputdir}{basename}/{FILENAMES[1]}", sep=",") # faked
        #print(f"\n****ERROR!  No DBPEDIA files {auxx}\n\n")
        #data_dbpedia.rename(columns={"BIO-LINGUAKIT": "BIO3"}, inplace=True)
        print(f"\n **** Error = {e}\nCONTINUING")
        continue


    # Initialize DataFrame for merged results
    df_result = pd.DataFrame(columns=["WORD", "BIO1", "BIO2", "BIO3", "FILE", "LINE", "POSITION"])

    # Debugging file setup
    if DEBUG:
        debug_file = open("debug-join.txt", "w", encoding="utf-8")



    

    # Ensure required columns exist
    required_columns = ["WORD", "BIO1", "FILE", "LINE", "POSITION"]
    for col in required_columns:
        if col not in data_allen.columns:
            raise ValueError(f"Missing required column '{col}' in data_allen.")

    # Merge AllenNLP and Linguakit results
    print("Merging AllenNLP and Linguakit results...")
    df_result = align_words(
        data_allen, data_linguakit, "BIO1", "BIO2", "FILE", "LINE", "POSITION"
    )

    # Merge DBpedia results
    print("Merging DBpedia results...")
    dbpedia_dict = dict(zip(data_dbpedia["WORD"], data_dbpedia["BIO3"]))

    df_result["BIO3"] = df_result["WORD"].map(lambda word: dbpedia_dict.get(word, "O"))

    # Clean BIO3 column
    df_result["BIO3"] = df_result["BIO3"].replace(
        {"[A-Za-z]+;": "", "U-": "B-", "L-": "I-"}, regex=True
    ).replace(
        {"Person$": "PER", "Place$": "LOC", "Organisation$": "ORG"}, regex=True
    )

    # Save the output
    output_file = f"{outputdir}{basename}/output.join"
    print(f"NER alignment completed. Output will be saved in '{output_file}'.")
    
    df_result.to_csv(output_file, index=False)
    if DEBUG:
        debug_file.close()

    #print(f"NER alignment completed. Output saved as '{output_file}'.")


# In[ ]:






/home/ematos/phd/pipeline_eurlex/INITIAL_NERS
/home/ematos/phd/pipeline_eurlex
output dir = /home/ematos/phd/pipeline_eurlex/INITIAL_NERS/OUTPUT_RESULTS//


Processing files:   0%|                                | 0/35 [00:00<?, ?file/s]


Processing legal_001.txt
file Allen = /home/ematos/phd/pipeline_eurlex/INITIAL_NERS/OUTPUT_RESULTS//legal_001/output.allen.csv
file Linguakit = /home/ematos/phd/pipeline_eurlex/INITIAL_NERS/OUTPUT_RESULTS//legal_001/output.linguakit
       WORD       BIO2           FILE
0   DECISÃO          O  legal_001.txt
1        DA  B-NP00SP0  legal_001.txt
2  COMISSÃO  L-NP00SP0  legal_001.txt
file dbpedia = /home/ematos/phd/pipeline_eurlex/INITIAL_NERS/OUTPUT_RESULTS//legal_001/output.dbpediaNER
Merging AllenNLP and Linguakit results...


Processing files:   6%|█▎                      | 2/35 [00:01<00:18,  1.79file/s]

Merging DBpedia results...
NER alignment completed. Output will be saved in '/home/ematos/phd/pipeline_eurlex/INITIAL_NERS/OUTPUT_RESULTS//legal_001/output.join'.

Processing legal_002.txt
file Allen = /home/ematos/phd/pipeline_eurlex/INITIAL_NERS/OUTPUT_RESULTS//legal_002/output.allen.csv
file Linguakit = /home/ematos/phd/pipeline_eurlex/INITIAL_NERS/OUTPUT_RESULTS//legal_002/output.linguakit
          WORD       BIO2           FILE
0  REGULAMENTO          O  legal_002.txt
1            (          O  legal_002.txt
2          CEE  U-NP00G00  legal_002.txt
file dbpedia = /home/ematos/phd/pipeline_eurlex/INITIAL_NERS/OUTPUT_RESULTS//legal_002/output.dbpediaNER
Merging AllenNLP and Linguakit results...
Merging DBpedia results...
NER alignment completed. Output will be saved in '/home/ematos/phd/pipeline_eurlex/INITIAL_NERS/OUTPUT_RESULTS//legal_002/output.join'.

Processing legal_003.txt
file Allen = /home/ematos/phd/pipeline_eurlex/INITIAL_NERS/OUTPUT_RESULTS//legal_003/output.allen.csv
f

...

Merging DBpedia results...
NER alignment completed. Output will be saved in '/home/ematos/phd/pipeline_eurlex/INITIAL_NERS/OUTPUT_RESULTS//legal_032/output.join'.

Processing legal_033.txt
file Allen = /home/ematos/phd/pipeline_eurlex/INITIAL_NERS/OUTPUT_RESULTS//legal_033/output.allen.csv
file Linguakit = /home/ematos/phd/pipeline_eurlex/INITIAL_NERS/OUTPUT_RESULTS//legal_033/output.linguakit
          WORD       BIO2           FILE
0  REGULAMENTO          O  legal_033.txt
1            (          O  legal_033.txt
2           CE  U-NP00O00  legal_033.txt
file dbpedia = /home/ematos/phd/pipeline_eurlex/INITIAL_NERS/OUTPUT_RESULTS//legal_033/output.dbpediaNER
Merging AllenNLP and Linguakit results...


Processing files:  97%|██████████████████████▎| 34/35 [00:24<00:00,  1.98file/s]

Merging DBpedia results...
NER alignment completed. Output will be saved in '/home/ematos/phd/pipeline_eurlex/INITIAL_NERS/OUTPUT_RESULTS//legal_033/output.join'.

Processing legal_034.txt
file Allen = /home/ematos/phd/pipeline_eurlex/INITIAL_NERS/OUTPUT_RESULTS//legal_034/output.allen.csv
file Linguakit = /home/ematos/phd/pipeline_eurlex/INITIAL_NERS/OUTPUT_RESULTS//legal_034/output.linguakit
          WORD       BIO2           FILE
0  REGULAMENTO          O  legal_034.txt
1            (          O  legal_034.txt
2           CE  U-NP00O00  legal_034.txt
file dbpedia = /home/ematos/phd/pipeline_eurlex/INITIAL_NERS/OUTPUT_RESULTS//legal_034/output.dbpediaNER
Merging AllenNLP and Linguakit results...
Merging DBpedia results...
NER alignment completed. Output will be saved in '/home/ematos/phd/pipeline_eurlex/INITIAL_NERS/OUTPUT_RESULTS//legal_034/output.join'.

Processing legal_035.txt
file Allen = /home/ematos/phd/pipeline_eurlex/INITIAL_NERS/OUTPUT_RESULTS//legal_035/output.allen.csv
f

Processing files: 100%|███████████████████████| 35/35 [00:29<00:00,  1.17file/s]

Merging DBpedia results...
NER alignment completed. Output will be saved in '/home/ematos/phd/pipeline_eurlex/INITIAL_NERS/OUTPUT_RESULTS//legal_035/output.join'.


## STEP5 - Create Decision File

In [2]:
#!/usr/bin/env python
# coding: utf-8

# In[1]:


#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
NER Data Processing Script
Created for robust and efficient handling of NER outputs with alignment and consensus building.
"""

import os
import pandas as pd
import sys
from tqdm import tqdm
from collections import Counter


# In[2]:


import os

working_dir = os.getcwd()
print(working_dir)

parts = working_dir.split("/")
if parts[-1].startswith("pipeline"): ### BUG SOLVED MAY 2025
    pipeline = working_dir
else:
    pipeline ="/".join(parts[:-1])
print(pipeline)


# In[3]:


#base = "/home/ematos/phd/pt/INITIAL_NERs/"
sys.path.append(pipeline)

# Configure directories
try:
    from config.configsite import * #filesdir, linguakitdir, inputdir, outputdir, basedir
except ImportError:
    print("Error: 'configsite' module not found. Ensure it is correctly configured.")
    sys.exit(1)
filesdir = f"{pipeline}/{dataset}_PROCESSED/"
lista_path = filesdir + "files_list.csv"
print(f"Using files list = {lista_path}")

lista = pd.read_csv(lista_path)
print(lista.head())

outputdir=pipeline+"INITIAL_NERS/"+resultsdir+"/"
print(f"output dir = {outputdir}") # new AT april 2025


# In[4]:


def wta(tag_str, threshold=2):
    """
    Determines the most common tag based on a threshold.
    """
    tags = tag_str.split(";")
    cnt = Counter(tags)
    most_common, count = cnt.most_common(1)[0]

    # Return "O" if no tag meets the threshold
    if count < threshold:
        return "O"
    return f"B-{most_common}" if most_common != "O" else "O"


# In[7]:


#####################################################################
debug = True
# Iterate through each file in the list
for _, row in tqdm(lista.iterrows(), total=len(lista), desc="Processing files", unit="file"):
    filename = row.get("filename")
    basename = row.get("basename")

    # Load the input DataFrame
    filename = "output.join"
    df = pd.read_csv(f"{outputdir}{basename}/{str(filename)}", sep=",")
    if debug:
        auxx = f"{outputdir}{basename}/{str(filename)}"
        print(f"Reading from {auxx}")

    # Initialize a new DataFrame for additional processing
    df2 = pd.DataFrame({
        "TAGS": ["O"] * len(df),        # Aggregated tags
        "TYPES": ["0"] * len(df),       # Entity types
        "NTAGS": [0] * len(df),         # Number of non-"O" tags
        "ENTITY": ["O"] * len(df),      # Entity labels (BIO format)
        "WTA": ["O"] * len(df),         # Winner Takes All label
        "CONSENSUS": ["O"] * len(df)    # Final consensus labels
    })

    # Map Linguakit entity codes to meaningful labels
    entity_mapping = {
        "NP00G00": "LOC",
        "NP00SP0": "PER",
        "NP00V00": "MISC",
        "NP00O00": "ORG"
    }
    # Replace entity codes in BIO2 column
    df["BIO2"] = df["BIO2"].replace(entity_mapping, regex=True)

    # Standardize BIO formats in BIO2 and BIO1 columns
    df["BIO2"] = df["BIO2"].replace({"U-": "B-"}, regex=True)
    df["BIO1"] = df["BIO1"].replace({"U-": "B-"}, regex=True)

    # Combine tags from BIO1, BIO2, and optionally BIO3
    df2["TAGS"] = df["BIO1"] + ";" + df["BIO2"] + ";" + df.get("BIO3", "O") + ";"

    # Extract types by removing BIO prefixes
    df2["TYPES"] = df2["TAGS"].replace({"[UBIL]-": ""}, regex=True)

    # Calculate NTAGS (number of non-"O" tags)
    df2["NTAGS"] = df2["TAGS"].apply(lambda x: 3 - x.count("O;"))

    # Assign ENTITY = "ENT" if BIO1, BIO2, or BIO3 is not "O"
    df["ENTITY"] = df.apply(
        lambda row: "ENT" if row["BIO1"] != "O" or row["BIO2"] != "O" or row["BIO3"] != "O" else "O",
        axis=1
    )

    # Apply WTA logic to TYPES column
    df2["WTA"] = df2["TYPES"].apply(lambda x: wta(x, threshold=2))

    # Merge results into the original DataFrame
    df["CONSENSUS"] = df2["WTA"]
    df["WTA"] = df2["WTA"]

    # Define the correct headers
    headers = [
        "WORD", "BIO1", "BIO2", "FILE", "LINE", "POSITION",
        "BIO3", "CONSENSUS", "WTA", "ENTITY"
    ]

    # Ensure all columns are present and properly ordered
    df = df.reindex(columns=headers)

    # Save the processed file with headers
    output_path = os.path.join(outputdir, str(basename), "output.decision")
    df.to_csv(output_path, sep="\t", index=False, header=True)

    # Save in CoNLL format (without headers)
    conll_path = output_path.replace(".decision", ".conll")
    df.to_csv(conll_path, sep=" ", index=False, header=False)

    # Append the results to the global decision.final file
    final_output_path = os.path.join(outputdir, "decision_final.csv")

    # Write header only if the file does not already exist
    if not os.path.exists(final_output_path):
        df.to_csv(final_output_path, sep="\t", index=False, header=True, mode="w")
    else:
        df.to_csv(final_output_path, sep="\t", index=False, header=False, mode="a")

print(f"Processing complete.\n Results saved in each folder and accumulated in '{final_output_path}'.")


# In[ ]:





# In[ ]:






/home/ematos/phd/pipeline_eurlex/INITIAL_NERS
/home/ematos/phd/pipeline_eurlex
Using files list = /home/ematos/phd/pipeline_eurlex//DATASET_EURLEX_PROCESSED/files_list.csv
        filename   basename
0  legal_001.txt  legal_001
1  legal_002.txt  legal_002
2  legal_003.txt  legal_003
3  legal_004.txt  legal_004
4  legal_005.txt  legal_005
output dir = /home/ematos/phd/pipeline_eurlex/INITIAL_NERS/OUTPUT_RESULTS//


Processing files:   0%|                                | 0/35 [00:00<?, ?file/s]

Reading from /home/ematos/phd/pipeline_eurlex/INITIAL_NERS/OUTPUT_RESULTS//legal_001/output.join


Processing files:   6%|█▎                      | 2/35 [00:00<00:01, 18.39file/s]

Reading from /home/ematos/phd/pipeline_eurlex/INITIAL_NERS/OUTPUT_RESULTS//legal_002/output.join
Reading from /home/ematos/phd/pipeline_eurlex/INITIAL_NERS/OUTPUT_RESULTS//legal_003/output.join
Reading from /home/ematos/phd/pipeline_eurlex/INITIAL_NERS/OUTPUT_RESULTS//legal_004/output.join


...

Reading from /home/ematos/phd/pipeline_eurlex/INITIAL_NERS/OUTPUT_RESULTS//legal_026/output.join
Reading from /home/ematos/phd/pipeline_eurlex/INITIAL_NERS/OUTPUT_RESULTS//legal_027/output.join
Reading from /home/ematos/phd/pipeline_eurlex/INITIAL_NERS/OUTPUT_RESULTS//legal_028/output.join
Reading from /home/ematos/phd/pipeline_eurlex/INITIAL_NERS/OUTPUT_RESULTS//legal_029/output.join
Reading from /home/ematos/phd/pipeline_eurlex/INITIAL_NERS/OUTPUT_RESULTS//legal_030/output.join
Reading from /home/ematos/phd/pipeline_eurlex/INITIAL_NERS/OUTPUT_RESULTS//legal_031/output.join


Processing files:  94%|█████████████████████▋ | 33/35 [00:02<00:00, 17.70file/s]

Reading from /home/ematos/phd/pipeline_eurlex/INITIAL_NERS/OUTPUT_RESULTS//legal_032/output.join
Reading from /home/ematos/phd/pipeline_eurlex/INITIAL_NERS/OUTPUT_RESULTS//legal_033/output.join
Reading from /home/ematos/phd/pipeline_eurlex/INITIAL_NERS/OUTPUT_RESULTS//legal_034/output.join
Reading from /home/ematos/phd/pipeline_eurlex/INITIAL_NERS/OUTPUT_RESULTS//legal_035/output.join


Processing files: 100%|███████████████████████| 35/35 [00:02<00:00, 13.36file/s]

Processing complete.
 Results saved in each folder and accumulated in '/home/ematos/phd/pipeline_eurlex/INITIAL_NERS/OUTPUT_RESULTS//decision_final.csv'.


## STEP 6 - Rearrange Decision to Bert

In [3]:
#!/usr/bin/env python
# coding: utf-8

# In[1]:


#!/usr/bin/env python
# coding: utf-8

# Import necessary libraries
import requests
import re
import spacy
from spacy.tokens import Doc
from spacy import displacy
from pathlib import Path
import pandas as pd
import os
import pandas as pd
import sys
from tqdm import tqdm
from collections import Counter


# In[2]:


import os

working_dir = os.getcwd()
print(working_dir)

parts = working_dir.split("/")
#if parts[-1] == "pipeline":
if parts[-1].startswith("pipeline"): ### BUG SOLVED MAY 2025
    pipeline = working_dir
else:
    pipeline ="/".join(parts[:-1])
print(pipeline)


# In[3]:


#-----------------------------------------------------------------------------------
#base = "/home/ematos/phd/NER/process1/process_script/"
sys.path.append(pipeline)

# Configure directories
try:
    from config.configsite import * #filesdir, linguakitdir, inputdir, outputdir, basedir
except ImportError:
    print("Error: 'configsite' module not found. Ensure it is correctly configured.")
    sys.exit(1)

#-----------------------------------------------------------------------------------


# In[4]:


outputdir=pipeline+"INITIAL_NERS/"+resultsdir+"/"
print(f"output dir = {outputdir}") # new AT april 2025


# In[5]:


# Load SpaCy English model
nlp = spacy.load("en_core_web_sm")

# Define helper function to preprocess BIO annotations
def preprocess_bio_tags(entities):
    """Convert BIO tags to a consistent format."""
    processed_entities = []
    for item in entities:
        if 'U-' in item:
            item = item.replace('U-', 'B-')
        if 'L-' in item:
            item = item.replace('L-', 'I-')
        processed_entities.append(item)
    return processed_entities

# Load CSV file
filename = outputdir+"decision_final.csv"
df = pd.read_csv(filename,sep='\t')
df['linha'] = df['FILE']+'_'+df['LINE'].astype(str)


# In[6]:


#Step 1: Extract unique values
unique_values = df['linha'].unique()

# Step 2: Create a mapping from unique values to integers
value_to_int = {value: idx + 1 for idx, value in enumerate(unique_values)}

# Step 3: Apply the mapping to create a new column
df['mapped_value'] = df['linha'].map(value_to_int)

#df.columns = ['FILE','LINE','WORD', 'BIO', 'BIO1', 'BIO2', 'BIO3', 'WTA', 'ENTITY']
df['Sentence #'] = 'Sentence:'+df['mapped_value'].astype(str)

df['POS'] = 'X'

# Apply the preprocess_bio_tags function to specific columns
columns_to_process = ['BIO1', 'BIO2', 'BIO3','CONSENSUS','WTA']

for col in columns_to_process:
    df[col] = df[col].apply(lambda x: " ".join(preprocess_bio_tags(x.split())) if isinstance(x, str) else x)

df


# In[7]:


df["BIO"] = df["CONSENSUS"]
df.head(70)


# In[13]:


def update_bio(row):
    if row['BIO'] == 'O':
        for col in ('BIO1', 'BIO2', 'BIO3'):
            if row[col] != 'O' and pd.notna(row[col]):
                return row[col]  # Return the first non-'O' value
    return row['BIO']  # Keep original BIO if no changes
    
df['BIO'] = df.apply(update_bio, axis=1)
df


# In[14]:


def to_entity(label):
    if label == 'O':
        return label
    prefix, _ = label.split('-', 1)
    return f'{prefix}-ENTITY'

# Apply transformation to the BIO column
df['BIO'] = df['BIO'].apply(to_entity)


# In[15]:


df['BIO'].unique()


# In[16]:


def insert_blank_lines(df):
    """Insert a blank line whenever the LINE number changes."""
    new_rows = []
    prev_line = None  # Track the previous LINE value

    for i, row in df.iterrows():
        if prev_line is not None and row['LINE'] != prev_line:
            # Insert a blank row before the current row
            new_rows.append(pd.Series({col: '' for col in df.columns}))

        new_rows.append(row)
        prev_line = row['LINE']  # Update previous LINE value

    # Convert back to DataFrame
    return pd.DataFrame(new_rows, columns=df.columns)

# Apply the function
df = insert_blank_lines(df)
df


# In[17]:


df2 =df[['Sentence #','WORD','BIO','POS','LINE','POSITION','FILE']].copy()
df2.rename(columns={'WORD':'Word','BIO':'Tag'},inplace=True)


# In[18]:


df2


# In[19]:


df2.to_csv(f'{outputdir}/decision_entity_bert_arrange.csv', sep='\t')


# In[20]:


## ??? NEEDE FOR WAHT ??

# commented AT, april 2025
#df2.to_csv(f'/home/ematos/phd/pt/ENTITY_DETECTOR_BY_BERT/INPUT_DATASET/decision_entity_bert_arrange.csv', sep='\t')


# In[ ]:






/home/ematos/phd/pipeline_eurlex/INITIAL_NERS
/home/ematos/phd/pipeline_eurlex
output dir = /home/ematos/phd/pipeline_eurlex/INITIAL_NERS/OUTPUT_RESULTS//
